# frog / non_frog 分類モデルの学習 (Keras / TensorFlow)

**Artificial Sound Creatures** — カエル鳴き声識別 CNN の訓練ノートブック。

このノートブックは Mac 2 台プロトタイプ用の学習パスです。学習した `frog_cnn.h5` を
`frog_node.py --model frog_cnn.h5` に渡すとリアルタイム推論に使えます。

## 最重要ルール: 特徴抽出は 1 つの実装を共有する

「学習時と推論時でメルスペクトログラムがズレて、実機だけ精度が壊れる」のが
このプロジェクト最大の落とし穴。それを防ぐため、特徴抽出は `features.py` の
`logmel` を**学習・推論・(将来の)実機で共有**する。このノートブックも
`features.py` を import して使う（コピペで別実装を作らない）。

## 実行環境

- ローカル(`prototype/` 内): このノートブックと `features.py` を同じ場所に置く。
- Google Colab: 左のファイルペインに `features.py` をアップロードしてから実行。

## データ配置

```
data/
  frog/       *.wav   # アマガエル。実機スピーカー再生→マイク録り直しを主軸に
  non_frog/   *.wav   # 会場の暗騒音・人声・足音・音楽など
```
各クラス 100 個以上を推奨。16kHz モノラルが理想（それ以外でも読み込み時に整える）。

In [ ]:
# 0. 準備 --------------------------------------------------------------
# 必要なら次を有効化（Colab など）:
# !pip install tensorflow numpy matplotlib

import glob
import os
import wave

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers as L

print("TensorFlow", tf.__version__)

In [ ]:
# 1. 特徴抽出は features.py を共有（別実装を作らない！）-----------------
try:
    from features import FeatureParams, logmel, mel_filterbank
except ImportError as e:
    raise SystemExit(
        "features.py が見つかりません。prototype/features.py をこのノートブックと"
        "同じ場所に置く（Colab ならアップロードする）こと。"
    ) from e

P = FeatureParams(clip_seconds=1.0)   # 1 判定あたりの窓長
FB = mel_filterbank(P)
print(f"入力形状 (frames x mels) = {P.n_frames} x {P.n_mels}")
print(f"sample_rate={P.sample_rate}, window_ms={P.window_ms}, hop_ms={P.hop_ms}")

In [ ]:
# 2. データ読み込み ----------------------------------------------------
CLASSES = ["non_frog", "frog"]   # index 0,1（classifier.py の解釈と一致）
DATA_DIR = "./data"


def read_wav(path, sr):
    with wave.open(path, "rb") as w:
        n, sw, ch = w.getnframes(), w.getsampwidth(), w.getnchannels()
        raw = w.readframes(n)
    dtype = {1: np.int8, 2: np.int16, 4: np.int32}[sw]
    x = np.frombuffer(raw, dtype=dtype).astype(np.float32)
    if ch > 1:
        x = x.reshape(-1, ch).mean(axis=1)
    return x / (np.max(np.abs(x)) or 1.0)


def load_dataset(data_dir, p, fb):
    X, y, paths = [], [], []
    for label, cls in enumerate(CLASSES):
        files = sorted(glob.glob(os.path.join(data_dir, cls, "*.wav")))
        for path in files:
            X.append(logmel(read_wav(path, p.sample_rate), p, fb))
            y.append(label)
            paths.append(path)
        print(f"{cls}: {len(files)} files")
    if not X:
        raise SystemExit(f"wav が見つかりません: {data_dir}/{{{','.join(CLASSES)}}}/")
    X = np.stack(X)[..., np.newaxis].astype(np.float32)
    return X, np.asarray(y), paths


X, y, paths = load_dataset(DATA_DIR, P, FB)
print("X:", X.shape, " クラス分布:", dict(zip(CLASSES, np.bincount(y, minlength=2))))

In [ ]:
# 3. 特徴量を目視確認（各クラス 1 枚）----------------------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, cls in zip(axes, [1, 0]):     # frog, non_frog
    hits = np.where(y == cls)[0]
    if len(hits) == 0:
        continue
    ax.imshow(X[hits[0], ..., 0].T, origin="lower", aspect="auto")
    ax.set_title(CLASSES[cls])
    ax.set_xlabel("frame")
    ax.set_ylabel("mel bin")
plt.tight_layout()
plt.show()

In [ ]:
# 4. 学習用 / テスト用に分割 ------------------------------------------
rng = np.random.default_rng(0)
idx = rng.permutation(len(X))
X, y = X[idx], y[idx]

split = int(len(X) * 0.8)
Xtr, Xte = X[:split], X[split:]
ytr, yte = y[:split], y[split:]
print("train:", Xtr.shape, " test:", Xte.shape)

In [ ]:
# 5. 軽量 DS-CNN を定義 -----------------------------------------------
# Depthwise-Separable Conv で軽量化。int8 量子化前提の素直な構成。
def build_model(input_shape, n_classes=2):
    return tf.keras.Sequential([
        L.Input(shape=input_shape),
        L.Conv2D(16, 3, padding="same", activation="relu"),
        L.BatchNormalization(),
        L.MaxPooling2D(2),
        L.SeparableConv2D(32, 3, padding="same", activation="relu"),
        L.BatchNormalization(),
        L.MaxPooling2D(2),
        L.SeparableConv2D(64, 3, padding="same", activation="relu"),
        L.BatchNormalization(),
        L.GlobalAveragePooling2D(),
        L.Dropout(0.3),
        L.Dense(n_classes, activation="softmax"),
    ])


model = build_model(X.shape[1:], len(CLASSES))
model.summary()

In [ ]:
# 6. 学習 --------------------------------------------------------------
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

hist = model.fit(Xtr, ytr,
                 validation_data=(Xte, yte),
                 epochs=30, batch_size=16)

In [ ]:
# 7. 評価（学習曲線 + 混同行列）---------------------------------------
plt.plot(hist.history["accuracy"], label="train")
plt.plot(hist.history["val_accuracy"], label="val")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend(); plt.title("accuracy")
plt.show()

pred = model.predict(Xte, verbose=0).argmax(axis=1)
acc = float((pred == yte).mean())
print(f"test accuracy = {acc:.3f}")

cm = np.zeros((2, 2), dtype=int)
for t, p_ in zip(yte, pred):
    cm[t, p_] += 1
print("confusion matrix (行=正解, 列=予測)  [non_frog, frog]:")
print(cm)

# 注意: データが少ないと accuracy が高く出ても過学習の可能性。
# 会場想定の環境音 non_frog と、スピーカー再生を録り直した frog を十分入れて再検証すること。

In [ ]:
# 8. 書き出し（Keras .h5 と int8 TFLite）------------------------------
model.save("frog_cnn.h5")


def representative_dataset():
    for i in range(min(200, len(Xtr))):
        yield [Xtr[i:i + 1]]


conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = representative_dataset
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
with open("frog_cnn_int8.tflite", "wb") as f:
    f.write(conv.convert())

print("saved: frog_cnn.h5, frog_cnn_int8.tflite")

## 次のステップ

1. **Mac で推論**: `python frog_node.py --model frog_cnn.h5` で 1 台稼働 → 2 台で逆相を検証。
2. **精度が出ないとき**: データを増やす（特に会場想定の `non_frog` と、スピーカー再生を
   録り直した `frog`）。`epochs` や層の幅を調整。
3. **ESP32 へ**: 実機はこの Keras 経路をそのまま載せるより、**Edge Impulse** に同じ収録データを
   アップロードして Arduino ライブラリを書き出す方が、C++ の特徴抽出を書かずに済み確実。
   このノートブックで掴んだ「必要なデータ量・モデル規模の勘」がそのまま活きる。

> メモ: このノートブックの学習は Mac 2 台プロトタイプ（挙動検証）用。実機本番のモデルは
> Edge Impulse 側で作り直してよい（共通資産はコードではなく **データと特徴仕様**）。